# Local MedGemma Testing

This notebook tests the MedGemma inference locally using the `MedGemmaWoundAnalyzer` wrapper. It enables loading the model and running predictions on wound images.

In [ ]:
# Install dependencies if missing
# !pip install torch torchvision torchaudio transformers accelerate pillow matplotlib

In [ ]:
# Setup and Imports
%load_ext autoreload
%autoreload 2

import os
import sys
import torch
from PIL import Image
import matplotlib.pyplot as plt
from transformers import AutoProcessor, PaliGemmaForConditionalGeneration, AutoTokenizer, AutoModelForCausalLM

# Add src to path
sys.path.append(os.path.abspath('src'))

from medgemma_inference import MedGemmaWoundAnalyzer

In [ ]:
# Configuration
IMAGE_PATH = "sample_wound.jpg"
MODEL_ID_4B = "google/paligemma-3b-mix-224" # Using public PaliGemma as proxy for MedGemma 4B
MODEL_ID_27B = "google/gemma-2-2b-it" # Small Gemma for classification testing (proxy for 27B)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

In [ ]:
# Load 4B Model (VLM) for Description
print(f"Loading VLM: {MODEL_ID_4B}...")

try:
    processor_4b = AutoProcessor.from_pretrained(MODEL_ID_4B)
    model_4b = PaliGemmaForConditionalGeneration.from_pretrained(
        MODEL_ID_4B, 
        torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32
    ).to(DEVICE)
    print("VLM Loaded.")
except Exception as e:
    print(f"Failed to load VLM: {e}")
    model_4b = None
    processor_4b = None

In [ ]:
# Initialize Analyzer with Real Models (pass None to verify Mock fallback if needed)
analyzer = MedGemmaWoundAnalyzer(
    model_4b=model_4b,
    processor_4b=processor_4b
)

In [ ]:
# Run Analysis on Sample Image
if os.path.exists(IMAGE_PATH):
    # Show image
    img = Image.open(IMAGE_PATH)
    plt.imshow(img)
    plt.axis('off')
    plt.show()

    # Run inference (set use_mock=False to use the loaded model)
    print("Running inference...")
    try:
        result = analyzer.analyze_wound_image(IMAGE_PATH, use_mock=False)
        print("\n--- Analysis Result ---")
        import json
        print(json.dumps(result, indent=2))
    except Exception as e:
        print(f"Inference failed: {e}")
else:
    print(f"Image not found at {IMAGE_PATH}")